# Train Test Creator

## Import Libraries

In [1]:
import json
import os
import sys

import joblib
import numpy as np
import pandas as pd
from dotenv import load_dotenv
from sklearn.preprocessing import StandardScaler

sys.path.insert(0, os.path.abspath(".."))

from dtos.tabular_database_driver_dtos.postgre_sql_connection_dto import PostgreSQLConnectionDto
from logger.logger import Logger
from tabular_database_driver.postgre_sql_driver import PostgreSQLDriver
from utils.constants import DATABASE_MAIN_V2, UNIFIED_SCHEMA

load_dotenv()

True

## Parameters

In [2]:
TICKER = "vcb"
TARGET_HORIZON = 5  # matches UNIFIED_TARGET_HORIZON used to build the target

PG_HOST = os.getenv("POSTGRES_HOST", "localhost")
PG_PORT = int(os.getenv("POSTGRES_PORT", 5432))
PG_USER = os.getenv("POSTGRES_USER", "postgres")
PG_PASSWORD = os.getenv("POSTGRES_PASSWORD", "")

LOOKBACK_DAY = 20
TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
# TEST_RATIO = 0.15 (implicit)

DROP_COLUMNS = ["exchange", "ticker", "date"]
TARGET_COLUMN = "target"
SCALER_TAG = "std"  # std = StandardScaler

# Recognizable folder name carrying the main settings, e.g.
#   vcb_lb20_h5_tr70_val15_test15_std
TEST_RATIO = round(1 - TRAIN_RATIO - VAL_RATIO, 4)
DATASET_NAME = (
    f"{TICKER.lower()}"
    f"_lb{LOOKBACK_DAY}"
    f"_h{TARGET_HORIZON}"
    f"_tr{int(TRAIN_RATIO * 100)}"
    f"_val{int(VAL_RATIO * 100)}"
    f"_test{int(TEST_RATIO * 100)}"
    f"_{SCALER_TAG}"
)
OUTPUT_DIR = os.path.join("../train_test_set", DATASET_NAME)
print(f"Dataset name: {DATASET_NAME}")

Dataset name: vcb_lb20_h5_tr70_val15_test15_std


## Load Data

In [3]:
logger = Logger(file_name="../../logs/train_test_creator")

driver = PostgreSQLDriver(logger=logger)
driver.connect(
    PostgreSQLConnectionDto(
        logger=logger,
        host=PG_HOST,
        user=PG_USER,
        password=PG_PASSWORD,
        port=PG_PORT,
        database=DATABASE_MAIN_V2,
    )
)

df = driver.select(
    schema_name=UNIFIED_SCHEMA,
    table_name=f"unified_{TICKER.lower()}",
    order_by=["date"],
)

driver.disconnect()

print(f"Loaded {len(df)} rows, {len(df.columns)} columns")
df

Loaded 4213 rows, 1018 columns


,exchange,ticker,date,open,high,low,close,volume,close_bb_20_upper,close_bb_20_middle,...,is_quarter_start,is_quarter_end,is_year_end,month_sin,month_cos,day_of_week_sin,day_of_week_cos,day_of_year_sin,day_of_year_cos,target
0,HOSE,VCB,2009-06-30,12637.379,12637.379,12637.379,12637.379,1396178,NaN,NaN,...,0,1,0,1.224647e-16,-1.000000,0.781832,0.623490,0.025818,-0.999667,-5.833330
1,HOSE,VCB,2009-07-01,13269.249,13269.249,12532.067,12742.690,29666217,NaN,NaN,...,1,0,0,-5.000000e-01,-0.866025,0.974928,-0.222521,0.008607,-0.999963,-8.264456
2,HOSE,VCB,2009-07-02,12532.067,12637.379,12110.822,12216.134,7196119,NaN,NaN,...,0,0,0,-5.000000e-01,-0.866025,0.433884,-0.900969,-0.008607,-0.999963,-6.896552
3,HOSE,VCB,2009-07-03,11900.199,12005.511,11794.888,11794.888,4271697,NaN,NaN,...,0,0,0,-5.000000e-01,-0.866025,-0.433884,-0.900969,-0.025818,-0.999667,-8.035719
4,HOSE,VCB,2009-07-06,11794.888,12321.445,11794.888,12321.445,7462326,NaN,NaN,...,0,0,0,-5.000000e-01,-0.866025,0.000000,1.000000,-0.077386,-0.997001,-16.239320
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4208,HOSE,VCB,2026-06-02,62300.000,62700.000,61600.000,61600.000,8255897,65659.695,62250.0,...,0,0,0,1.224647e-16,-1.000000,0.781832,0.623490,0.486273,-0.873807,NaN
4209,HOSE,VCB,2026-06-03,61700.000,62500.000,61400.000,61900.000,4438645,65639.400,62320.0,...,0,0,0,1.224647e-16,-1.000000,0.974928,-0.222521,0.471160,-0.882048,NaN
4210,HOSE,VCB,2026-06-04,61800.000,62500.000,61700.000,62200.000,4072125,65603.910,62415.0,...,0,0,0,1.224647e-16,-1.000000,0.433884,-0.900969,0.455907,-0.890028,NaN
4211,HOSE,VCB,2026-06-05,62400.000,62500.000,61600.000,61700.000,4018358,65575.164,62465.0,...,0,0,0,1.224647e-16,-1.000000,-0.433884,-0.900969,0.440519,-0.897743,NaN


## Preprocess Data

In [4]:
# Drop tail rows where target is NaN (future return not yet available)
df_clean = df.dropna(subset=[TARGET_COLUMN]).reset_index(drop=True)
print(f"After dropping NaN targets: {len(df_clean)} rows (dropped {len(df) - len(df_clean)})")

dates = pd.to_datetime(df_clean["date"])

# Separate features and target
feature_df = df_clean.drop(columns=DROP_COLUMNS + [TARGET_COLUMN])
target_series = df_clean[TARGET_COLUMN]

# Forward-fill then back-fill any remaining NaN in features (early rolling-window indicators)
nan_before = feature_df.isna().sum().sum()
feature_df = feature_df.ffill().bfill()
nan_after = feature_df.isna().sum().sum()
print(f"Feature NaN filled: {nan_before} -> {nan_after}")

# Identify already-bounded columns that should NOT be scaled:
#   - cyclical encodings (sin/cos in [-1, 1])
#   - binary 0/1 flags (TA crossovers, band flags, calendar flags)
cyclical_cols = [c for c in feature_df.columns if c.endswith("_sin") or c.endswith("_cos")]
binary_cols = [
    c for c in feature_df.columns
    if set(feature_df[c].dropna().unique()).issubset({0, 1, 0.0, 1.0})
]
bounded_cols = sorted(set(cyclical_cols) | set(binary_cols))
scale_cols = [c for c in feature_df.columns if c not in set(bounded_cols)]

print(f"Feature matrix shape: {feature_df.shape}")
print(f"Bounded (not scaled): {len(bounded_cols)}  |  Continuous (scaled): {len(scale_cols)}")

After dropping NaN targets: 4208 rows (dropped 5)
Feature NaN filled: 62195 -> 0


Feature matrix shape: (4208, 1014)
Bounded (not scaled): 228  |  Continuous (scaled): 786


## Split Indices (chronological)

In [5]:
n_rows = len(feature_df)
train_end = int(n_rows * TRAIN_RATIO)
val_end = int(n_rows * (TRAIN_RATIO + VAL_RATIO))

print(f"Total rows         : {n_rows}")
print(f"Train rows [0:{train_end}]   {dates.iloc[0].date()} -> {dates.iloc[train_end - 1].date()}")
print(f"Val   rows [{train_end}:{val_end}]   {dates.iloc[train_end].date()} -> {dates.iloc[val_end - 1].date()}")
print(f"Test  rows [{val_end}:{n_rows}]   {dates.iloc[val_end].date()} -> {dates.iloc[n_rows - 1].date()}")

Total rows         : 4208
Train rows [0:2945]   2009-06-30 -> 2021-05-07
Val   rows [2945:3576]   2021-05-10 -> 2023-11-10
Test  rows [3576:4208]   2023-11-13 -> 2026-06-01


## Normalize (fit on train rows only)

In [6]:
# Order columns as [scaled continuous ... | bounded] and remember the order
ordered_cols = scale_cols + bounded_cols
feat = feature_df[ordered_cols].copy()

# --- Feature scaler: fit on TRAIN rows only, transform the whole array ---
feature_scaler = StandardScaler()
feature_scaler.fit(feat.iloc[:train_end][scale_cols].values)
feat[scale_cols] = feature_scaler.transform(feat[scale_cols].values)

# --- Target scaler: fit on TRAIN rows only ---
target_scaler = StandardScaler()
target_scaler.fit(target_series.iloc[:train_end].values.reshape(-1, 1))
target_scaled = target_scaler.transform(target_series.values.reshape(-1, 1)).ravel()

X_arr = feat.values.astype(np.float32)
y_arr = target_scaled.astype(np.float32)

print(f"Scaled feature array: {X_arr.shape}  (cols 0:{len(scale_cols)} scaled, rest bounded)")
print(f"Target scaled  mean~0: {y_arr[:train_end].mean():.4f}  std~1: {y_arr[:train_end].std():.4f}")

Scaled feature array: (4208, 1014)  (cols 0:786 scaled, rest bounded)
Target scaled  mean~0: 0.0000  std~1: 1.0000


## Create Windows (3D tensors)

In [7]:
def make_windows(X, y, start, end):
    """Sliding windows of length LOOKBACK_DAY; target is the value at the window's last day.

    Each split starts LOOKBACK_DAY-1 rows early so its first window is complete
    without borrowing target rows from the previous split (no leakage —
    features are scaled with train-only statistics, targets only look forward).
    """
    xs, ys = [], []
    for i in range(start, end - LOOKBACK_DAY + 1):
        xs.append(X[i : i + LOOKBACK_DAY])
        ys.append(y[i + LOOKBACK_DAY - 1])
    return np.stack(xs).astype(np.float32), np.array(ys, dtype=np.float32)

X_train, y_train = make_windows(X_arr, y_arr, 0, train_end)
X_val,   y_val   = make_windows(X_arr, y_arr, train_end - (LOOKBACK_DAY - 1), val_end)
X_test,  y_test  = make_windows(X_arr, y_arr, val_end - (LOOKBACK_DAY - 1), n_rows)

print(f"X_train : {X_train.shape}  y_train : {y_train.shape}")
print(f"X_val   : {X_val.shape}  y_val   : {y_val.shape}")
print(f"X_test  : {X_test.shape}  y_test  : {y_test.shape}")

X_train : (2926, 20, 1014)  y_train : (2926,)
X_val   : (631, 20, 1014)  y_val   : (631,)
X_test  : (632, 20, 1014)  y_test  : (632,)


## Save Model-Ready Datasets

In [8]:
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Tensors
np.save(os.path.join(OUTPUT_DIR, "X_train.npy"), X_train)
np.save(os.path.join(OUTPUT_DIR, "y_train.npy"), y_train)
np.save(os.path.join(OUTPUT_DIR, "X_val.npy"), X_val)
np.save(os.path.join(OUTPUT_DIR, "y_val.npy"), y_val)
np.save(os.path.join(OUTPUT_DIR, "X_test.npy"), X_test)
np.save(os.path.join(OUTPUT_DIR, "y_test.npy"), y_test)

# Scalers (joblib handles sklearn objects well)
joblib.dump(feature_scaler, os.path.join(OUTPUT_DIR, "feature_scaler.pkl"))
joblib.dump(target_scaler, os.path.join(OUTPUT_DIR, "target_scaler.pkl"))

# Metadata
metadata = {
    "dataset_name": DATASET_NAME,
    "ticker": TICKER.lower(),
    "schema": UNIFIED_SCHEMA,
    "table": f"unified_{TICKER.lower()}",
    "lookback_day": LOOKBACK_DAY,
    "target_horizon": TARGET_HORIZON,
    "target_column": TARGET_COLUMN,
    "dropped_columns": DROP_COLUMNS,
    "n_features": len(ordered_cols),
    "feature_columns": ordered_cols,
    "scaled_columns": scale_cols,
    "bounded_columns": bounded_cols,
    "split_ratios": {"train": TRAIN_RATIO, "val": VAL_RATIO, "test": TEST_RATIO},
    "shapes": {
        "X_train": list(X_train.shape), "y_train": list(y_train.shape),
        "X_val": list(X_val.shape),     "y_val": list(y_val.shape),
        "X_test": list(X_test.shape),   "y_test": list(y_test.shape),
    },
    "date_ranges": {
        "train": [str(dates.iloc[0].date()), str(dates.iloc[train_end - 1].date())],
        "val":   [str(dates.iloc[train_end].date()), str(dates.iloc[val_end - 1].date())],
        "test":  [str(dates.iloc[val_end].date()), str(dates.iloc[n_rows - 1].date())],
    },
    "scaler": {"feature": "StandardScaler", "target": "StandardScaler"},
    "nan_handling": {"target": "dropped tail NaN rows", "features": "ffill+bfill"},
    "created_at": pd.Timestamp.now().strftime("%Y-%m-%d"),
}
with open(os.path.join(OUTPUT_DIR, "metadata.json"), "w") as f:
    json.dump(metadata, f, indent=2)

print(f"Saved to {OUTPUT_DIR}")
for fname in sorted(os.listdir(OUTPUT_DIR)):
    size = os.path.getsize(os.path.join(OUTPUT_DIR, fname)) / 1e6
    print(f"  {fname:24s} {size:8.2f} MB")

Saved to ../train_test_set\vcb_lb20_h5_tr70_val15_test15_std
  X_test.npy                  51.27 MB
  X_train.npy                237.36 MB
  X_val.npy                   51.19 MB
  feature_scaler.pkl           0.02 MB
  metadata.json                0.06 MB
  target_scaler.pkl            0.00 MB
  y_test.npy                   0.00 MB
  y_train.npy                  0.01 MB
  y_val.npy                    0.00 MB
